# Crop-Specific Maize Monitoring — Greater Horn of Africa (all countries, all modules)
### One Colab notebook · Google Earth Engine · 11 countries · 2024

This notebook runs the **as-built ICPAC planting pipeline** for **any country and season** in the
Greater Horn of Africa, one module at a time, with a short explanation before each. It reuses the
exact `src/` code and the `build_product_image()` graph that produced the 2024 continental asset set,
so the demo here is the same computation as the operational run — only clipped to a small fast test box.

**The four modules**
1. **Planting window** — when the crop went in (onset), per pixel.
2. **Risk monitoring** — staged water balance (WRSI) and the water / heat / vegetation stresses.
3. **CPI & yield** — the Crop Performance Index and the calibrated yield estimate.
4. **Flooding / waterlogging** — the wet-side hazard the WRSI cannot see.

**Countries & seasons (2024):** Kenya (Long, Short) · Ethiopia (Meher, Belg) · Somalia (Gu, Deyr) ·
Uganda (1st, 2nd) · Rwanda (A, B) · Burundi (A, B) · Tanzania (Masika, Msimu, Vuli) · South Sudan (Main).

### How to run
1. Put the whole **`planting_pipeline`** folder on your Google Drive (so `import src` works).
2. Run the cells **top to bottom**. Section 0 signs you into Earth Engine and mounts Drive.
3. In the **config cell**, set `COUNTRY` and `SEASON` from the printed list, then run each module.

> Every module reuses your `src/` code from Drive. Maps use **geemap** with the native Earth Engine
> Layers panel (toggle + opacity), so no Google Maps API key is needed.

## Section 0 — Setup (Earth Engine + Drive)

### Stage 0 · What this notebook is

**The operational graph, run interactively.** It calls the same `build_product_image()` used for the
2024 continental asset set, so what you see here is the operational computation clipped to a small
box, not a simplified demo. Every module below reads pieces of **one** result, which is why they are
always internally consistent.

**Expected output.** `installed.`

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine

`EE ready: ok`. Switch to `indigo-proxy-484220-q8` before submitting the continental batch in
section 5; the queue is per project and this one has stalled.

In [ ]:
import ee
PROJECT="ee-manzikye"   # <-- your Earth Engine cloud project
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive

`pipeline on path: ...`. An `AssertionError` means the folder is not on Drive at that path.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if your folder is elsewhere
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not found at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Configure country & season
The pipeline knows every viable maize season from `config/season_calendar.csv`. The cell below prints
the full menu, then you pick one. Each country has a small **test box** so the demo runs in seconds;
set `USE_WHOLE_COUNTRY=True` to run the real AOI (slower). The onset method (green-up vs rainfall) and
any cross-year handling are chosen automatically by the calendar, exactly as in the operational run.

### Stage 0d · The product menu

**What this stage does.** Prints every viable maize (country, season) in
`config/season_calendar.csv`, and defines a small demo box per country. Sixteen products across eleven
countries were built for 2024.

**Viability is a judgement recorded in the calendar**, not a guess: a season marked low viability is
one where maize area is negligible or the season cannot be detected reliably. Only High and Medium
appear here.

**Expected output.** A list of pairs such as `Kenya Long rains`, `Ethiopia Meher`, `Somalia Gu`,
`Rwanda Season A`, `Tanzania Msimu`.

In [ ]:
from src import utils
from run_all_maize_2024 import build_product_image, YEAR
from run import GAUL_NAME
from src import zonal_aggregate as ZA
kc, soil = utils.load_crop_coeffs()
ROWS = {(r["country"], r["season"]): r for r in
        utils.viable_products(utils.load_calendar("config/season_calendar.csv"))
        if r["crop"].lower() == "maize"}
print("Available (country, season) products:")
for (c, s) in ROWS: print(f"   {c:12s} {s}")

# small representative demo boxes [west,south,east,north] (maize belts); fallback = country bounds
TEST_BOX = {
  "Kenya":       [34.4,-1.2,37.8,1.2],   "Ethiopia":  [37.0,7.0,39.5,9.5],
  "Somalia":     [42.0,1.5,44.5,3.5],    "Uganda":    [30.5,0.2,33.5,2.2],
  "Rwanda":      [29.2,-2.4,30.6,-1.3],  "Burundi":   [29.2,-3.9,30.5,-2.8],
  "Tanzania":    [34.0,-6.5,36.5,-4.5],  "South Sudan":[30.0,4.0,33.0,7.0],
}

### Stage 0e · Pick one product

**What to set.** `COUNTRY`, `SEASON`, and whether to run the real country boundary.

**`USE_WHOLE_COUNTRY = True` is a different order of job.** The demo box returns in seconds to
minutes. A whole country with the fused green-up runs for tens of minutes, and interactive
`getInfo()` calls will start timing out. At that point use the batch scripts in section 5.

**The onset method is chosen for you**, by the calendar. Second and short seasons route to the
rainfall-anchored rule because green-up detection fails there: in Rwanda and Burundi, Season A
green-up returned 64 usable pixels against 3,365 for the rainfall rule.

In [ ]:
# >>> EDIT THESE TWO <<<
COUNTRY = "Kenya"        # e.g. "Tanzania"
SEASON  = "Long rains"   # must match one printed above for this country, e.g. "Msimu"
USE_WHOLE_COUNTRY = False # True = real AOI (slow); False = fast test box

assert (COUNTRY, SEASON) in ROWS, f"{(COUNTRY, SEASON)} not in the menu above"
r = ROWS[(COUNTRY, SEASON)]
if USE_WHOLE_COUNTRY:
    gname = GAUL_NAME.get(COUNTRY) or GAUL_NAME.get(COUNTRY.replace(" ", "_"), COUNTRY)
    aoi_run = ZA.gaul_admin(ee, [gname], level=0).geometry()
else:
    aoi_run = ee.Geometry.Rectangle(TEST_BOX.get(COUNTRY, [34.4,-1.2,37.8,1.2]))

import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE"); m.centerObject(aoi_run, zoom); return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity); return m
print(f"Selected: {COUNTRY} · {SEASON} · {YEAR}  ·  onset window dekads = {r['sos_detection_window']}")

### Compute the product once (feeds Modules 1–3)
`build_product_image()` runs the whole graph for the selected country/season and returns the output
image plus every intermediate (`planting`, staged WRSI, the three stresses, CPI, yield). Modules 1–3
below just visualise pieces of this one result — so they are always internally consistent.

### Stage 1 · Compute the whole product once

**What this stage does.** Runs the entire graph and returns the output image plus every intermediate:
the planting dekad, the staged water balance, the three stresses, CPI and yield. Modules 1 to 3 below
only draw pieces of this.

**Expected output.** The band list, then the onset method that was selected. Bands include
`planting_dekad`, `CPI`, `yield_tha_x100` and, in the rich stack, six WRSI and WSI stage bands.

**Yield is stored times 100 as an integer.** `yield_tha_x100` divided by 100 is tonnes per hectare.
This keeps the asset small; forgetting it is the most common misreading of these assets.

**$Y_m$ here is the calibrated ceiling**, through `CPI.ym_img_for`, unlike the older single-country
notebook which hard-codes 6.0.

In [ ]:
out, aoi_run, M3 = build_product_image(ee, r, kc, soil, aoi=aoi_run, rich=True)
planting = M3["planting"]; staged = M3["staged"]
Sw, Sh, Sv = M3["Sw"], M3["Sh"], M3["Sv"]; cpi_img, yld = M3["cpi"], M3["yld"]; mask = M3["mask"]
print("computed:", out.bandNames().getInfo())
print("onset method:", M3["onset_method"])

## Module 1 — Planting window (onset)

**What it answers:** *for each field, which 10-day period (dekad) did the crop go in?*

**How it works.** For **main seasons** the onset is a **cue-fusion green-up**: Sentinel-2 red-edge
(NDRE) is the primary signal, MODIS FPAR confirms canopy build-up, and Sentinel-1 SAR fills cloud
gaps. The first sustained green-up crossing inside the season window is the start-of-season (SOS);
a crop-specific offset (maize 2 dekads, wheat/teff 1) converts SOS to the **planting dekad**. For
**short / second seasons** green-up is too weak, so onset is **rainfall-anchored** (CHIRPS onset:
25 mm in a dekad with ≥20 mm follow-on, and P/PET support). **Cross-year seasons** (e.g. Tanzania
Msimu, Dec→Feb) are built from the prior year with the dekads wrapped past 36.

*Dekad 1 = 1–10 Jan … dekad 36 = 21–31 Dec. Higher = later planting.*

### Module 1 · Planting window

**What it answers.** For each field, which ten-day period did the crop go in?

**Expected values.** Inside the season window for that country. Kenya long rains: modal dekad 8, with
most area in dekads 7 to 9. Against farmer reports for 2024, bias **−0.31 dekads**, MAE **1.02
dekads**, **93 %** of counties within two dekads at county level and **96 %** of 855 wards at ward
level. One dekad is the noise floor.

**The colour ramp here runs the full 1 to 36**, so a single-season map occupies a narrow slice of it.
That is expected; the module notebooks stretch the ramp to the season window instead.

In [ ]:
M = new_map()
ee_layer(M, planting.clip(aoi_run), {"min":1,"max":36,
         "palette":["08306b","08519c","2171b5","4292c6","6baed6","9ecae1","c6dbef"]}, f"Planting dekad — {COUNTRY} {SEASON}")
M

## Module 2 — Risk monitoring (staged WRSI + stresses)

**What it answers:** *where and at which growth stage is the crop under stress?*

**How it works.** From the planting dekad we run a **FAO-56/33 water balance** (WRSI) forward through
three stages — **vegetative → flowering → grain-fill** — using ERA5-Land ET₀ (Hargreaves), CHIRPS
rainfall, and SoilGrids water-holding capacity. Each stage yields a running **WRSI** (100 = no water
limitation) and a **water-stress index (WSI)**. Flowering is weighted most heavily (FAO-33 Ky = 1.5),
because water or heat stress there costs the most yield. Alongside water we compute:
- **Heat stress** — heat-degree-dekads above a maize threshold, concentrated at flowering (pollen sterility).
- **Vegetation stress** — a down-weighted VCI / FPAR-anomaly confirmation of canopy condition.

The layers below are on a **0–100** scale (stresses: higher = worse; WRSI: higher = better).

### Module 2 · Risk monitoring

**What it answers.** Where, and at which growth stage, is the crop under stress?

$$\mathrm{WRSI}=100\frac{\sum_t AET_t}{\sum_t WR_t},\qquad
AET_t=\min(SW_{t-1}+P_t,\,K_{c,t}ET_{0,t}).$$

The three stress layers are drawn as percentages so they can be compared with each other.

**Expected values.**

* **WRSI at flowering** 70 to 100 over maize in a normal season; below 50 is the FEWS crop-failure
  class; 50 to 60 poor, 60 to 80 mediocre, 80 to 95 mild, 95 and above unstressed.
* **$S_{\text{water}}$** carries almost all of the signal. It is the only one of the three with a
  calibrated parameter set behind it, the FAO-33 $K_y$ values.
* **$S_{\text{heat}}$ is zero in the highland seasons and that is correct.** The 33 °C cap is a
  dekad-mean; measured dekad-mean flowering maxima peak at 22.9, 25.9 and 29.3 °C across Kenya's three
  regimes. The term is live for lowland and Sahelian seasons.
* **$S_{\text{veg}}$** is capped at 0.4 by design. It confirms; it does not drive.

**Toggle the stress layers against WRSI.** If $S_{\text{water}}$ is high where WRSI is also high, the
stage weighting is doing the work: a deficit at flowering costs three times what the same deficit costs
during vegetative growth.

In [ ]:
M = new_map()
ee_layer(M, staged["wrsi_flo"].clip(aoi_run), {"min":0,"max":100,"palette":["d73027","fee08b","1a9850"]}, "WRSI — flowering (100=no stress)")
ee_layer(M, Sw.multiply(100).clip(aoi_run), {"min":0,"max":100,"palette":["ffffff","fdae61","d73027"]}, "Water stress S_water (%)", False)
ee_layer(M, Sh.multiply(100).clip(aoi_run), {"min":0,"max":100,"palette":["ffffff","fdae61","d73027"]}, "Heat stress S_heat (%)", False)
ee_layer(M, Sv.multiply(100).clip(aoi_run), {"min":0,"max":100,"palette":["ffffff","fdae61","d73027"]}, "Vegetation stress S_veg (%)", False)
M   # toggle layers in the Layers panel

## Module 3 — Crop Performance Index & yield

**What it answers:** *how good is the season, and how much maize per hectare?*

**How it works.** The three stresses combine **multiplicatively** (AquaCrop-style) into a single
**Crop Performance Index**:

$$\text{CPI} = 100\,(1-S_{water})(1-S_{heat})(1-S_{veg})$$

CPI is the fraction of the attainable ceiling the crop is on track to reach. Yield is then
$\text{yield} = (\text{CPI}/100)\times Y_m$, where **Yₘ is an attainable-ceiling** calibrated
against **HarvestStat Africa** official statistics (least-squares through origin, 70/30 train/test).
Yₘ is **AEZ-aware** where it helps: in Kenya's Long rains the cool **highland** (≥ 1800 m) carries a
higher ceiling (3.2 t/ha) than the rest (2.1 t/ha), because the highland edge is *potential*, not
water timing. Uncalibrated countries fall back to season defaults (6.0 t/ha main, 4.5 short).

### Module 3 · Crop performance index and yield

$$\mathrm{CPI}=100\,(1-S_{\text{water}})(1-S_{\text{heat}})(1-S_{\text{veg}}),
\qquad Y_a=\frac{\mathrm{CPI}}{100}\,Y_m.$$

**$Y_m$ is calibrated, per country and season**, fitted to HarvestStat sub-national yields with the
median year as the target and tested on a 70/30 split repeated 200 times: Kenya 2.34 long rains and
1.44 short rains, Ethiopia 4.14, Rwanda 2.61, Burundi 1.88, Somalia 1.02, Uganda 2.34 provisional.
Tanzania and South Sudan have no HarvestStat maize yields and fall back to 6.0, which every country
that could be tested shows to be several times too high; treat their yields as unusable in level.

**Where $r$ is near zero the ceiling fixes the level, not the ranking.** Rwanda, Burundi and Somalia
are level-only: use their yields for national and seasonal totals, not to rank districts.

In [ ]:
M = new_map()
ee_layer(M, cpi_img.clip(aoi_run), {"min":0,"max":100,"palette":["d73027","fee08b","1a9850"]}, "CPI (0-100)")
ee_layer(M, yld.clip(aoi_run), {"min":0,"max":6,"palette":["ffffcc","c2e699","78c679","238443"]}, "Yield (t/ha)", False)
M

### Module 3b · The two numbers to check

**Expected values over the demo box.**

| Country · season | Mean CPI | Mean yield t/ha |
|---|---|---|
| Kenya · Long rains | 55 to 85 | about 1.5 |
| Ethiopia · Meher | 60 to 90 | about 3 |
| Somalia · Gu | lower and more variable | about 1 |

**Two failure signatures.** A mean CPI near 100 means the stresses did not compute, usually because the
water balance ran on an empty planting image. A mean yield above 4 t/ha in Kenya means $Y_m$ fell
through to the uncalibrated default, which happens when the country or season string does not match
the calendar exactly.

Remember to divide `yield_tha_x100` by 100, which this cell does for you.

In [ ]:
# quick numbers over the test box (mean CPI / mean yield)
stats = out.select(["CPI","yield_tha_x100"]).reduceRegion(
    ee.Reducer.mean(), aoi_run, scale=250, maxPixels=int(1e10), bestEffort=True).getInfo()
print(f"{COUNTRY} {SEASON}:  mean CPI = {stats.get('CPI'):.1f}   mean yield = {stats.get('yield_tha_x100')/100:.2f} t/ha")

## Module 4 — Flooding / waterlogging (the wet-side hazard)

**What it answers:** *where is there too much water — the risk WRSI cannot see?*

WRSI only measures **deficit**; it saturates at 100 and is blind to excess. Two complementary metrics
cover the wet side:
- **SPI-3 wet anomaly** (validated) — a 3-month standardized-precipitation surplus flags surface /
  seasonal excess rainfall.
- **Soil aeration-stress index** (modelled, uncalibrated) — days the root zone sits above field
  capacity toward saturation during the crop cycle, from a SoilGrids + Saxton-Rawls hydrology.

Blue = wetter / more waterlogged. Read these together with the crop mask.

### Module 4 · The wet-side hazard

**Why it is separate.** WRSI caps soil water at field capacity and discards the rest, so excess water
is invisible to it by construction.

**SPI-3 wet, validated**: $\mathbf{1}[\mathrm{SPI}_3\ge1.5]$, a seasonal surface anomaly against 1981
to 2020. **Aeration stress, modelled and uncalibrated**: a daily root-zone balance from SoilGrids and
Saxton-Rawls, where stage-weighted stress accumulates only while the soil stays above the anaerobiosis
point and resets whenever it drains. Stage weights are reversed from the deficit side, veg 1.00, flo
0.60, grf 0.35, because young maize is the vulnerable stage.

**Expected values.** The waterlogging index is **zero over most pixels in most seasons**. Report the
SPI-3 wet layer; use the aeration layer to rank places, never as a calibrated magnitude.

In [ ]:
from src import excess as EX, soil as SOIL
mz = kc["maize"]; d_veg = mz["L_ini"]+mz["L_dev"]; d_flo = d_veg+mz["L_mid"]; lgp = mz["LGP_dekads"]
ss, se = utils.sos_window_dekads(r["sos_detection_window"])
end_m = 5 if SEASON=="Long rains" else (9 if SEASON=="Meher" else 12)
wet = EX.spi3_wet(ee, aoi_run, YEAR, end_month=end_m)
hy = SOIL.build_hydro_mm(ee, root_depth_cm=100)   # FC/SAT/tau from SoilGrids + Saxton-Rawls
wl = EX.aeration_stress_index(ee, aoi_run, planting, YEAR, d_veg, d_flo, lgp, ss, se%36 or se, hy["FC_mm"], hy["SAT_mm"], hy["tau"])
print("excess / waterlogging computed")

In [ ]:
M = new_map()
ee_layer(M, wet.updateMask(mask).clip(aoi_run), {"min":0,"max":1,"palette":["ffffff","3690c0"]}, "SPI-3 very wet (excess, validated)")
ee_layer(M, wl.updateMask(mask).clip(aoi_run), {"min":0,"max":40,"palette":["f7fbff","6baed6","08306b"]}, "Soil waterlogging (modelled, uncal.)", False)
M

## Section 5 — Reproduce the whole continent (batch)

The single-country cells above are for exploration. The **operational continental run** is driven by
two scripts (run them from a machine with the repo, not cell-by-cell — they submit Earth Engine batch
**asset exports** that run server-side for hours):

```bash
# 1) submit every viable 2024 maize product as a 250 m CPI/yield asset (cpi_<Country>_<Season>_2024)
EE_PROJECT=ee-manzikye python run_all_maize_2024.py --stage all --submit

# 1b) OR the richer stack (planting_dekad + 6 WRSI/WSI stage bands) that feeds the app panels
EE_PROJECT=ee-manzikye python run_all_maize_2024.py --stage all --rich --submit

# 2) calibrate the yield ceiling Ym against HarvestStat (70/30) and print paste-ready values
EE_PROJECT=ee-manzikye python calibrate_ym_all.py

# 3) watch the batch until it settles
EE_PROJECT=ee-manzikye python poll_tasks.py --watch 300
```

Onset is chosen per season automatically (green-up for main seasons, rainfall-anchored for short /
second seasons); cross-year seasons (e.g. Tanzania Msimu) are wrapped past dekad 36. Skip-guards mean
re-running only submits products that are not already an asset or in flight. See the accompanying
**ALL_COUNTRIES_2024 documentation** for datasets, algorithms step-by-step, the A/B tests, and how the
gaps were filled.